#### Analysis 1 

In [1]:
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
fold = 0
exp_path = "./experiments/troubleshooting/20260227_103603"
fold_path = os.path.join(exp_path, "fold_" + str(fold))

In [ ]:
# Confusion matrix
cm = pd.read_csv(os.path.join(fold_path, "confusion_matrix.csv"), index_col=0)

lbls = ["CN", "MCI", "AD"]
cm.index = lbls
cm.columns = lbls

cmap = sn.light_palette('seagreen', as_cmap=True)
sn.heatmap(cm/cm.sum().sum(), annot=True, cmap=cmap)


In [ ]:
# Diagnosis distribution

sn.set_theme(palette='pastel')
samples = pd.read_csv(os.path.join(exp_path, "samples.csv"))
diag_dist = samples.groupby("label")["PET"].nunique()

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%\n(n={count})'
    return inner

plt.pie(
    diag_dist,
    labels=["CN", "MCI", "AD"],
    startangle=90,
    wedgeprops=dict(width=0.5),
    autopct=autopct_format(diag_dist),
    pctdistance=0.7
)
plt.show()

In [ ]:
# Train, validation and test splits

sn.set_theme(palette='pastel')

splits = [ 4327, 1050, 586]
lbls = ["Training", "Validation", "Test"]

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        count = int(round(pct * total / 100.0))
        return f'{pct:.0f}%\n(n={count})'
    return inner

plt.pie(
    splits,
    labels=lbls,
    startangle=90,
    wedgeprops=dict(width=0.5),
    autopct=autopct_format(diag_dist),
    pctdistance=0.7
)
plt.show()


In [ ]:
# Confidence distributions 

preds = pd.read_csv(os.path.join(fold_path, "predictions.csv"))

lbls = ["CN", "MCI", "AD"]

fig, axes = plt.subplots(3,3, figsize=(15,15))

for i in range(3):
    for j in range(3):

        axes[i,j].set_title(f"true {lbls[i]}, pred { lbls[j] }")
        axes[i,j].hist( 
            preds[ (preds["target"] == i) & (preds["pred"] == j) ]["confidence"], 
            bins=np.linspace(0.4,1,20),
            alpha=0.9, 
            cumulative=False)
        
        axes[i,j].set_ylim([0,50])



In [ ]:
import torch
import json 
import yaml

from pkg.utils.instantiate import instantiate
from pkg.training.criterion import build_criterion

In [ ]:
# Read config 
with open(os.path.join(exp_path, "config.yaml")) as f:
    cfg = yaml.safe_load(f)

print(cfg)

In [ ]:
dm = instantiate(cfg["datamodule"])
dm.setup()
dm.set_fold(fold)

In [ ]:
model = instantiate(cfg["model"])
criterion = build_criterion(cfg["criterion"], train_labels=dm.train_labels)

model.set_criterion(criterion)
best_state = torch.load( os.path.join(fold_path, "model.ckpt"), weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(best_state)

In [ ]:
from tqdm import tqdm

model.eval()

preds = []
targets = []
confs = []

for batch in tqdm(dm.test_dataloader()):

    out = model.test_batch(batch, 0)
    preds.append(out['preds'])
    targets.append(out['targets'])
    confs.append(out['confs'])

# Compute global confusion matrix and classification report
preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
confs = torch.cat(confs).numpy()


### Analysis 2

In [1]:
from pkg.utils.report import experiment_report
from pkg.utils.reproducibility import repo_state
from pkg.utils.instantiate import instantiate
from pkg.data.datasets import ADNIDataset
from pkg.training.criterion import build_criterion

from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Subset, DataLoader
from tqdm import tqdm 

import os
import yaml
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [33]:
#def analyze_predictions(path, fold): 
path = "./experiments/fdg_mri_demog_complete_only/20260319_043211/"
fold = 4

# Read config 
with open(os.path.join(path, "config.yaml")) as f: 
    cfg = yaml.load(f, yaml.FullLoader)

# Read commit
with open(os.path.join(path, "commit.txt")) as f:
    commit = f.read().strip()

# Read patch 
with open(os.path.join(path, "patch.diff"), "rb") as f:
    patch = f.read()

# Read indices 
with open(os.path.join(path, "indices.json")) as f:
    indices = json.loads(f.read())        

# Create dataset with saved samples
ds_cfg = cfg["datamodule"]["args"]["data"]
ds = ADNIDataset(
    data_dir=ds_cfg["data_dir"],
    cached_samples= os.path.join(path, "samples.csv"),
    modalities={"PET-fdg":"", "MRI":""},
)
ds.setup()

# Load model 
with repo_state(commit, patch):
    model = instantiate(cfg["model"])

# Setup model 
model.set_criterion(nn.CrossEntropyLoss(weight=torch.tensor([1,1,1])))

# Load state dict 
with open( os.path.join(path, f"fold_{fold}", "model_stage_0.ckpt"), "rb") as f:
    best_state = torch.load(f, map_location=torch.device("cpu"))
model.load_state_dict(best_state)
model.eval()

Verifying scans available in dir...


PoE(
  (experts): ModuleList(
    (0-1): 2 x Expert(
      (net): Sequential(
        (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
        (1): GroupNorm(4, 8, eps=1e-05, affine=True)
        (2): Swish()
        (3): ResidualBlock(
          (residual): Sequential(
            (0): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (1): GroupNorm(8, 8, eps=1e-05, affine=True)
            (2): Swish()
            (3): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (4): GroupNorm(8, 8, eps=1e-05, affine=True)
            (5): SAM3d(
              (conv): Conv3d(2, 1, kernel_size=(7, 7, 7), stride=(1, 1, 1), padding=(3, 3, 3))
            )
          )
          (skip1): Conv3d(8, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1))
        )
        (4): ResidualBlock(
          (residual): Sequential(
            (0): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))


In [34]:
# Test dataset 
ds_test = Subset(ds, indices["test_idx"])

# Loader 
test_loader = DataLoader(ds_test, batch_size=1, shuffle=False)

In [36]:
# Test dataset 
ds_test = Subset(ds, indices["test_idx"])

# Loader 
test_loader = DataLoader(ds_test, batch_size=1, shuffle=False)

subjects = []
preds = []
targets = []
confs = []

with torch.no_grad():
    for i, batch in tqdm(enumerate(test_loader)):

        batch["mask"][:,0] = 0 # Disable MRI
        out = model.test_batch(batch,i)

        subjects.append(batch["subject"][0])
        preds.append(out["preds"].detach().cpu().view(-1))
        targets.append(out["targets"].detach().cpu().view(-1))
        confs.append(out["confs"].detach().cpu().view(-1))
        
# Compute global confusion matrix and classification report
preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
confs = torch.cat(confs).numpy()

cm = pd.DataFrame(confusion_matrix(targets, preds))
report = pd.DataFrame(classification_report(targets, preds, digits=4, output_dict=True)).T

print(cm)
print(report)

308it [04:13,  1.22it/s]

    0   1   2
0  48  27   2
1  51  70  33
2   3   9  65
              precision    recall  f1-score     support
0              0.470588  0.623377  0.536313   77.000000
1              0.660377  0.454545  0.538462  154.000000
2              0.650000  0.844156  0.734463   77.000000
accuracy       0.594156  0.594156  0.594156    0.594156
macro avg      0.593655  0.640693  0.603079  308.000000
weighted avg   0.610336  0.594156  0.586925  308.000000


In [12]:

true_label = 2
pred_label = 1
lbls = ["CN", "MCI", "AD"]

selected_subj = set({})
converted = set({})

df_dx = pd.read_csv("/project/aereditato/cestari/adni-mri-classification/data/preprocessing_multimodal/csv/adni/DXSUM_02Feb2026.csv")

for i,subj in enumerate(subjects): 
    if (targets[i] == true_label) and (preds[i] == pred_label):
        selected_subj.add(subj)

        if len(df_dx[ (df_dx['PTID'] == subj) & (df_dx["DIAGNOSIS"] == pred_label+1) ]) > 0:
            converted.add(subj)

print(f"{len(converted)}/{len(selected_subj)} = {len(converted) / len(selected_subj):.2%} of { lbls[true_label] } patients incorrectly predicted as { lbls[pred_label]} also have a { lbls[pred_label]} diagnosis at some point")

7/15 = 46.67% of AD patients incorrectly predicted as MCI also have a MCI diagnosis at some point
